In [8]:
BASE_PATH = r"C:\Users\MEL\Downloads\DATA\SEMANA 4\Proyect_Lab\Proyecto_lab"

In [9]:
# Importamos las librerías necesarias y cargamos el dataset crudo
import pandas as pd
import numpy as np
import os



In [10]:
filepath = r"C:\Users\MEL\Downloads\DATA\SEMANA 4\Proyect_Lab\Proyecto_lab\data\raw\trials_raw.csv"
df = pd.read_csv(filepath)
print(f"✅ Datos cargados: {df.shape}")
df.head()

✅ Datos cargados: (4000, 10)


,nct_id,title,status,phase,start_date,enrollment,sponsor,sponsor_class,conditions,countries
0,NCT06423690,First in Human Study for the Assessment of Saf...,RECRUITING,NaN,2024-11-01,15.0,Snipe Medical,INDUSTRY,Lung Cancer,United Kingdom; Spain; Israel
1,NCT00257790,The Tobramycin Study,COMPLETED,PHASE4,2001-09,210.0,Oslo University Hospital,OTHER,Neutropenia; Fever; Cancer,NaN
2,NCT00369590,VEGF Trap in Treating Patients With Recurrent ...,COMPLETED,PHASE2,2006-08,58.0,National Cancer Institute (NCI),NIH,Adult Anaplastic Astrocytoma; Adult Anaplastic...,United States
3,NCT00513097,Curbing Tobacco Use in Suburban and Rural Schools,COMPLETED,PHASE1,2006-07-25,1289.0,M.D. Anderson Cancer Center,OTHER,Tobacco Use Cessation,United States
4,NCT04501497,Prospective Multicenter Observational Study of...,COMPLETED,NaN,2020-08-21,1221.0,Chugai Pharmaceutical,INDUSTRY,Non-small Cell Lung Cancer; Extensive Disease ...,Japan


In [11]:
# Exploramos la estructura del dataset antes de limpiar
print("=== TIPOS DE DATOS ===")
print(df.dtypes)
print("\n=== VALORES NULOS ===")
print(df.isnull().sum())
print("\n=== ESTADÍSTICAS ===")
print(df.describe())

=== TIPOS DE DATOS ===
nct_id            object
title             object
status            object
phase             object
start_date        object
enrollment       float64
sponsor           object
sponsor_class     object
conditions        object
countries         object
dtype: object

=== VALORES NULOS ===
nct_id              0
title               0
status              0
phase            1666
start_date         40
enrollment         73
sponsor             0
sponsor_class       0
conditions          0
countries         319
dtype: int64

=== ESTADÍSTICAS ===
         enrollment
count  3.927000e+03
mean   3.312243e+03
std    1.422159e+05
min    0.000000e+00
25%    2.500000e+01
50%    6.100000e+01
75%    1.700000e+02
max    8.804863e+06


In [12]:
# Técnica 1: Eliminar duplicados por nct_id
before = len(df)
df = df.drop_duplicates(subset=["nct_id"], keep="first")
after = len(df)
print(f"✅ Duplicados eliminados: {before - after}")
print(f"   Filas: {before} → {after}")

✅ Duplicados eliminados: 0
   Filas: 4000 → 4000


In [13]:
# Técnica 2: Tratar valores nulos
df["enrollment"] = df["enrollment"].fillna(df["enrollment"].median())
text_cols = ["phase", "sponsor_class", "conditions", "countries"]
for col in text_cols:
    df[col] = df[col].fillna("Unknown")
print("✅ Nulos tratados")
print(df.isnull().sum())

✅ Nulos tratados
nct_id            0
title             0
status            0
phase             0
start_date       40
enrollment        0
sponsor           0
sponsor_class     0
conditions        0
countries         0
dtype: int64


In [14]:
# Técnica 3: Estandarizar valores categóricos
phase_map = {
    "PHASE1": "Phase 1", "PHASE2": "Phase 2",
    "PHASE3": "Phase 3", "PHASE4": "Phase 4",
    "EARLY_PHASE1": "Phase 1 (Early)", "NA": "Not Applicable"
}
status_map = {
    "RECRUITING": "Recruiting", "COMPLETED": "Completed",
    "TERMINATED": "Terminated", "NOT_YET_RECRUITING": "Not Yet Recruiting",
    "ACTIVE_NOT_RECRUITING": "Active, Not Recruiting",
    "WITHDRAWN": "Withdrawn"
}
df["phase"] = df["phase"].map(phase_map).fillna(df["phase"])
df["status"] = df["status"].map(status_map).fillna(df["status"])
print("✅ Strings estandarizados")
print(df["phase"].value_counts())
print(df["status"].value_counts())

✅ Strings estandarizados
phase
Unknown            1666
Phase 2            1036
Phase 1             816
Phase 3             327
Phase 4              85
Phase 1 (Early)      70
Name: count, dtype: int64
status
Completed                  1723
UNKNOWN                     661
Recruiting                  597
Terminated                  381
Active, Not Recruiting      267
Not Yet Recruiting          179
Withdrawn                   140
ENROLLING_BY_INVITATION      25
SUSPENDED                    10
NO_LONGER_AVAILABLE          10
APPROVED_FOR_MARKETING        5
AVAILABLE                     2
Name: count, dtype: int64


In [15]:
# Técnica 4: Normalizar fechas y extraer año
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df["start_year"] = df["start_date"].dt.year
df.loc[df["start_year"] < 1990, "start_year"] = np.nan
df.loc[df["start_year"] > 2025, "start_year"] = np.nan
print("✅ Fechas normalizadas")
print(df["start_year"].value_counts().sort_index().tail(10))

✅ Fechas normalizadas
start_year
2016.0     91
2017.0    163
2018.0    176
2019.0    223
2020.0    210
2021.0    273
2022.0    250
2023.0    239
2024.0    249
2025.0    208
Name: count, dtype: int64


In [16]:
# Técnica 5: Tratar outliers en enrollment
print(f"Antes — max: {df['enrollment'].max()}, media: {df['enrollment'].mean():.0f}")
df.loc[df["enrollment"] > 1_000_000, "enrollment"] = np.nan
df["enrollment"] = df["enrollment"].fillna(df["enrollment"].median())
df["enrollment"] = df["enrollment"].astype(int)
print(f"Después — max: {df['enrollment'].max()}, media: {df['enrollment'].mean():.0f}")
print("✅ Outliers tratados")

Antes — max: 8804863.0, media: 3253
Después — max: 1000000, media: 1052
✅ Outliers tratados


In [17]:
# Técnica 6: Extraer país principal
df["primary_country"] = df["countries"].str.split(";").str[0].str.strip()
df["primary_country"] = df["primary_country"].replace("", "Unknown")
print("✅ País principal extraído")
print(df["primary_country"].value_counts().head(10))

✅ País principal extraído
primary_country
United States     1487
China              627
Unknown            319
France             305
Canada             121
Italy              104
South Korea         95
United Kingdom      93
Germany             77
Netherlands         60
Name: count, dtype: int64


In [18]:
# Técnica 7: Limpiar espacios en columnas de texto
df["title"] = df["title"].str.strip()
df["sponsor"] = df["sponsor"].str.strip()
print("✅ Columnas limpias")
print(f"Shape final: {df.shape}")
df.head()

✅ Columnas limpias
Shape final: (4000, 12)


,nct_id,title,status,phase,start_date,enrollment,sponsor,sponsor_class,conditions,countries,start_year,primary_country
0,NCT06423690,First in Human Study for the Assessment of Saf...,Recruiting,Unknown,2024-11-01,15,Snipe Medical,INDUSTRY,Lung Cancer,United Kingdom; Spain; Israel,2024.0,United Kingdom
1,NCT00257790,The Tobramycin Study,Completed,Phase 4,NaT,210,Oslo University Hospital,OTHER,Neutropenia; Fever; Cancer,Unknown,NaN,Unknown
2,NCT00369590,VEGF Trap in Treating Patients With Recurrent ...,Completed,Phase 2,NaT,58,National Cancer Institute (NCI),NIH,Adult Anaplastic Astrocytoma; Adult Anaplastic...,United States,NaN,United States
3,NCT00513097,Curbing Tobacco Use in Suburban and Rural Schools,Completed,Phase 1,2006-07-25,1289,M.D. Anderson Cancer Center,OTHER,Tobacco Use Cessation,United States,2006.0,United States
4,NCT04501497,Prospective Multicenter Observational Study of...,Completed,Unknown,2020-08-21,1221,Chugai Pharmaceutical,INDUSTRY,Non-small Cell Lung Cancer; Extensive Disease ...,Japan,2020.0,Japan


In [19]:
# Guardar dataset limpio
BASE_PATH = r"C:\Users\MEL\Downloads\DATA\SEMANA 4\Proyect_Lab\Proyecto_lab"
df.to_csv(f"{BASE_PATH}\\data\\clean\\trials_clean.csv", index=False)
print(f"✅ Datos limpios guardados")
print(f"Shape final: {df.shape}")

✅ Datos limpios guardados
Shape final: (4000, 12)
